In [1]:
import numpy as np
import pandas as pd
import random
import time
from rapidfuzz import process, fuzz, distance
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import spoa 
from sklearn.metrics.pairwise import pairwise_distances
from sklearn.cluster import HDBSCAN
import statistics
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import DBSCAN
import numpy as np
from scipy.cluster.hierarchy import linkage, fcluster, to_tree
from scipy.spatial.distance import pdist
import matplotlib.pyplot as plt

# --- 2. DATA GENERATION & UTILS ---
def mutate_sequence(seq, error_rate=0.10):
    if error_rate <= 0: return seq
    seq_list = list(seq)
    new_seq = []
    for base in seq_list:
        if random.random() < error_rate:
            r = random.random()
            if r < 0.5: new_seq.append(random.choice("ACGT")) 
            elif r < 0.75: 
                new_seq.append(base)
                new_seq.append(random.choice("ACGT"))
            else: pass 
        else:
            new_seq.append(base)
    return "".join(new_seq)

def gen_dna(k): return "".join(random.choices("ACGT", k=k))

# (Generating fresh data to ensure variables exist for the loop below)
n_items = 20_000 
pool_bc = [gen_dna(12) for _ in range(n_items)]
pool_ins = [gen_dna(random.randint(40,100)) for _ in range(n_items)]
n_repeat_bc = 500
pool_bc += pool_bc[0:n_repeat_bc]
pool_ins += [gen_dna(random.randint(40,100)) for _ in range(n_repeat_bc)]
pool_bc += pool_bc[0:n_repeat_bc]
pool_ins += [gen_dna(random.randint(40,100)) for _ in range(n_repeat_bc)]

data = []
for i in range(len(pool_bc)):
     n_reads = random.randint(2, 10) 
     for _ in range(n_reads):
         data.append({
             "ID": i,
             "Barcode": mutate_sequence(pool_bc[i], 0.05), 
             "Insert": mutate_sequence(pool_ins[i], 0.05)
         })

df = pd.DataFrame(data)

In [2]:
from scipy.stats import mode

def encode_msa(msa_strings, alphabet="ACGTN-"):
    char_to_int = {c: i for i, c in enumerate(alphabet)}
    max_len = max(len(s) for s in msa_strings)
    msa_padded = [s.ljust(max_len, '-') for s in msa_strings]
    msa_array = np.array([list(seq) for seq in msa_padded])
    N, L = msa_array.shape
    msa_int = np.zeros((N, L), dtype=np.int8)
    for char, idx in char_to_int.items():
        msa_int[msa_array == char] = idx
    return msa_int, len(alphabet)


def get_marginals(msa_int, vocab_size):
    # Create on hot matrix
    one_hot = np.eye(vocab_size)[msa_int]
    # Calculate frequencies of each base/N/- at each position in MSA
    P_i = one_hot.mean(axis=0)
    return one_hot, P_i

def compute_mi_scores(one_hot, P_i):
    """
    Compute mutual information scores in the MSA
    """
    N, L, A = one_hot.shape
    flat_view = one_hot.transpose(1, 2, 0).reshape(L * A, N)
    joint_probs_flat = (flat_view @ flat_view.T) / N
    P_ij = joint_probs_flat.reshape(L, A, L, A)
    P_product = P_i[:, :, None, None] * P_i[None, None, :, :]
    mask = P_ij > 0
    mi_matrix = np.zeros_like(P_ij)
    mi_matrix[mask] = P_ij[mask] * np.log(P_ij[mask] / P_product[mask])
    return mi_matrix.sum(axis=(1, 3))


def get_elbow_columns(mi_matrix, exclusion_distance=5, verbose=0):
    L = mi_matrix.shape[0]
    mask = np.triu(np.ones((L, L), dtype=bool), k=exclusion_distance + 1)
    rows, cols = np.where(mask)
    scores = mi_matrix[rows, cols]
    
    if len(scores) == 0: return np.array([])

    # 1. Sort Descending
    sorted_indices = np.argsort(scores)[::-1]
    sorted_scores = scores[sorted_indices]
    
    # 2. Determine "Signal End" (Noise Floor Truncation)
    # We stop analyzing where the score drops below the bottom 25% percentile
    noise_floor = np.percentile(scores, 25)
    
    # Keep points strictly ABOVE noise floor for the shape analysis
    # (But keep at least 5 points to allow for geometry, unless total is small)
    valid_mask = sorted_scores > noise_floor
    n_signal_points = np.sum(valid_mask)
    
    # Safety: if signal is super short, take at least top 10% or min 5
    min_points = min(len(sorted_scores), 5)
    cutoff_idx = max(n_signal_points, min_points)
    
    # 3. Truncate for Geometry Calculation
    # We only look for the elbow within the "Signal" region
    curve_y = sorted_scores[:cutoff_idx]
    n_points = len(curve_y)
    
    if n_points < 3:
        # Fallback for tiny signals
        return np.unique(np.concatenate([rows[sorted_indices[:n_points]], cols[sorted_indices[:n_points]]]))

    # 4. Standard Kneedle on Truncated Curve
    x_norm = np.linspace(0, 1, n_points)
    y_norm = (curve_y - curve_y.min()) / (curve_y.max() - curve_y.min() + 1e-9)
    
    # Line from First Point (0, 1) to Last Signal Point (1, 0)
    line_vec = np.array([1.0, -1.0]) # Direction vector (x=1-0, y=0-1)
    line_vec = line_vec / np.linalg.norm(line_vec)
    
    # Vectors from start (0, 1) to all points
    vec_from_start = np.stack([x_norm, y_norm - 1.0], axis=1)
    
    # Cross product (2D) to find distance
    distances = np.abs(vec_from_start[:, 0] * line_vec[1] - vec_from_start[:, 1] * line_vec[0])
    
    # 5. Select
    elbow_idx = np.argmax(distances)
    
    # If the plateau is perfectly flat, elbow_idx might be 0. 
    # In that case, we likely want the whole plateau, not just index 0.
    # We check if the point at elbow_idx is significantly higher than end.
    if elbow_idx == 0:
        # Heuristic: If index 0 is picked, but index 1 is very close in value, walk forward
        # This handles the "perfect plateau" case
        for i in range(1, n_points - 1):
            if curve_y[i] >= (curve_y[0] * 0.95): # Still within 5% of max
                elbow_idx = i
            else:
                break

    n_selected = elbow_idx + 1 # +1 because index is 0-based
    
    # --- Debug Plot (Only if Verbose) ---
    if verbose >= 3:
        import matplotlib.pyplot as plt
        plt.figure(figsize=(6, 3))
        plt.plot(range(n_points), curve_y, '-o', markersize=3, label='Signal Curve')
        plt.axvline(elbow_idx, color='r', linestyle='--', label=f'Elbow (k={n_selected})')
        plt.axhline(noise_floor, color='k', linestyle=':', alpha=0.5, label='Noise Floor')
        plt.title(f"Elbow Analysis (Selecting {n_selected} cols)")
        plt.legend()
        plt.show()

    selected_indices = sorted_indices[:n_selected]

    selected_columns = np.unique(np.concatenate([rows[selected_indices], cols[selected_indices]]))
    
    if verbose >= 3:
        print(f"selected MSA columns: {selected_columns}")
    return selected_columns


def calculate_msa_mi_and_top_cols(barcodes, inserts, verbose=0, mi_exclusion_distance=3):
    
    _, msa_barcodes = spoa.poa(barcodes, algorithm=2) 
    _, msa_inserts = spoa.poa(inserts, algorithm=2)
    msa_strings = [b + "-"*1 + i for b, i in zip(msa_barcodes, msa_inserts)]

    len_msa_inserts = len(msa_inserts[0])  # store for later checks

    # Encode MSA into a numpy array with numbers (should be treated as categoricals)
    msa_int, vocab_size = encode_msa(msa_strings)

    # Create one hot matrix and the frequencies of each base at each position in MSA
    one_hot, P_i = get_marginals(msa_int, vocab_size)
    
    # Calculate mutual information scores in MSA
    mi_matrix = compute_mi_scores(one_hot, P_i)

    # Use elbow approach to find pairs in MSA with high mutual information
    # Exclude very close positions as sequencing errors of more than one base may 
    # produce spurious mutual info
    top_cols = get_elbow_columns(mi_matrix, mi_exclusion_distance, verbose=verbose)   

    return msa_int, mi_matrix, top_cols, len_msa_inserts


def plot_dendrogram(model, **kwargs):
    # Create linkage matrix and then plot the dendrogram

    # create the counts of samples under each node
    counts = np.zeros(model.children_.shape[0])
    n_samples = len(model.labels_)
    for i, merge in enumerate(model.children_):
        current_count = 0
        for child_idx in merge:
            if child_idx < n_samples:
                current_count += 1  # leaf node
            else:
                current_count += counts[child_idx - n_samples]
        counts[i] = current_count

    # The linkage matrix has the format [idx1, idx2, dist, sample_count]
    linkage_matrix = np.column_stack([model.children_, model.distances_,
                                      counts]).astype(float)

    # Plot the corresponding dendrogram
    dendrogram(linkage_matrix, **kwargs)



def cluster_msa_subset(msa_subset, expected_error_rate, error_rate_multiplier=None, jump_thresh=None):

    assert (error_rate_multiplier is None) + (jump_thresh is None) == 1
    
    # Compute Hamming distance matrix
    dist_matrix_subset = pairwise_distances(msa_subset, metric='hamming')

    if dist_matrix_subset.shape[0] == 1:
        return dist_matrix_subset, np.array([0])  # only one
    # elif dist_matrix_subset.shape[0] == 2:
    #     return dist_matrix_subset, np.array([0, 1])  # just split in 2
    
    if error_rate_multiplier is not None: 
        # Then we are clustering based on expected error rate
        d_thresh = expected_error_rate * error_rate_multiplier
        
    else:
        # Now are are going to attempt an additional split, based on the hierarchy of the dendrogram
        # Create a copy to avoid modifying the original matrix
        temp_matrix = dist_matrix_subset.copy() 
        
        # Set the diagonal to infinity (a very large number)
        np.fill_diagonal(temp_matrix, np.inf) 
        
        # Find the minimum of the entire matrix
        min_dist = np.min(temp_matrix)
        
        d_thresh = min_dist * jump_thresh + expected_error_rate

        
    agg = AgglomerativeClustering(
    metric="precomputed",
    linkage="single",
    distance_threshold = d_thresh,
    n_clusters=None)
    
    labels = agg.fit_predict(dist_matrix_subset) 

    unique_labels = list(set(labels))

    if PLOTS:
        # --- 1. Heatmap ---
        plt.figure(figsize=(8, 7))
        sns.heatmap(
            dist_matrix_subset,
            annot=True,
            cmap="viridis",
            fmt=".2f",
            linewidths=.5,
            cbar_kws={'label': 'Hamming Distance'}
        )
        plt.title('Heatmap of Distance Matrix Subset', fontsize=14)
        plt.ylabel('Row Sample', fontsize=12)
        plt.xlabel('Column Sample', fontsize=12)
        plt.tight_layout()
        plt.show()

        agg.fit(dist_matrix_subset)
        
        # 3. Setup the plot
        plt.figure(figsize=(12, 6))
        plt.title('Hierarchical Clustering Dendrogram')
        plt.xlabel("Sample Index (or Cluster Size)")
        plt.ylabel("Distance (Average Linkage)")
        
        # 4. Plot the dendrogram
        # 'truncate_mode' condenses the plot if you have many points. 
        # Remove it to see the full tree.
        plot_dendrogram(agg, color_threshold=d_thresh)
        
        # 5. Visualise the decision threshold
        plt.axhline(y=d_thresh, c='r', ls='--', lw=2)
        
        plt.show()

    return dist_matrix_subset, labels



def cluster_barcode_insert_pairs(barcodes, inserts, mi_exclusion_distance, expected_error_rate, 
                                 error_rate_multiplier=None, jump_thresh=None, verbose=0, subset_msa=True):
    
    # Produce integer-encoded msa, mutual information matrix and identify most informative columns
    msa_int, mi_matrix, top_cols, len_msa_inserts = calculate_msa_mi_and_top_cols(barcodes, inserts, verbose, mi_exclusion_distance)

    if subset_msa:
        # Get just the most informative (i.e., highest mutual information) columns of MSA
        msa_subset = msa_int[:, top_cols]
    else:
        msa_subset = msa_int  # NO LONGER SUBSETTING!!! Useful if basing on overall error rate

    if PLOTS:
        # Set up the figure size
        plt.figure(figsize=(10, 4)) 
        
        # Use a specific color map (e.g., 'magma', 'viridis', or a custom one)
        sns.heatmap(
            msa_subset,
            cmap="viridis",  
            cbar=True,         # Show the color bar
            yticklabels=False, # Hide individual sequence labels if you have many
            xticklabels=10     # Show tick labels every 10 positions
        )
        
        plt.title("Heatmap of Integer Encoded MSA", fontsize=14)
        plt.ylabel("Sequence Index", fontsize=12)
        plt.xlabel("Position in Alignment", fontsize=12)
        plt.show()

    # Identify rough clusters using HDBSCAN
    dist_matrix_subset, labels = cluster_msa_subset(msa_subset, expected_error_rate, error_rate_multiplier, jump_thresh)

    return msa_int, mi_matrix, msa_subset, dist_matrix_subset, labels, len_msa_inserts


def plot_confusion_matrix(labels, real_IDs):
    from sklearn.metrics.cluster import contingency_matrix
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np
    
    # 1. Compute the contingency matrix
    # This counts the overlap between every unique value in labels vs real_IDs
    cont_mat = contingency_matrix(labels, real_IDs)
    
    # 2. Visualize
    plt.figure(figsize=(10, 8))
    sns.heatmap(cont_mat, annot=True, fmt='d', cmap="Blues")
    plt.xlabel("Real IDs (Arbitrary)")
    plt.ylabel("Predicted Labels (Arbitrary)")
    plt.title("Contingency Matrix (Unsorted)")
    plt.show()
    


import numpy as np
from scipy.stats import mode

import numpy as np

def fast_consensus(msa_list, threshold=0.5):
    """
    Rapid consensus that handles ties by lowercasing the winner.
    Uses int8 views for maximum speed and robustness.
    """
    if not msa_list:
        return ""

    # 1. Convert to BYTES (S1) and then VIEW as Integers (int8)
    # This is the key to speed: we work with numbers (ASCII codes), not strings.
    # 'A' is 65, 'a' is 97, '-' is 45.
    arr = np.array([list(s) for s in msa_list], dtype='S1')
    arr_int = arr.view(np.int8)
    
    n_seqs, length = arr_int.shape
    consensus_int = []
    
    # Constants for checks
    GAP_ASCII = 45 # '-'
    TO_LOWER_OFFSET = 32 # Add this to Uppercase ASCII to get Lowercase
    
    # 2. Iterate columns (Positions)
    # Using np.unique on int8 is extremely fast (much faster than strings)
    for i in range(length):
        col = arr_int[:, i]
        vals, counts = np.unique(col, return_counts=True)
        
        # Find the max vote count
        max_count = counts.max()
        
        # Identify how many characters share this max count (Ties)
        # valid_winners are the ASCII codes that tied for first place
        winners = vals[counts == max_count]
        
        # 3. Determine the Winner
        # Just pick the first one for now (if tie, we modify it later)
        best_char_code = winners[0] 
        
        # Check for Ambiguity (Tie)
        is_ambiguous = len(winners) > 1
        
        # 4. Filter Logic
        # Skip if the winner is a Gap
        if best_char_code != GAP_ASCII:
            
            # Check threshold
            if (max_count / n_seqs) >= threshold:
                
                # 5. Apply Lowercase if Ambiguous
                if is_ambiguous:
                    # e.g., 'A'(65) + 32 = 'a'(97)
                    # We ensure we don't lowercase non-letters just in case
                    if 65 <= best_char_code <= 90: 
                        best_char_code += TO_LOWER_OFFSET
                
                consensus_int.append(best_char_code)

    # 6. Convert list of integers back to string
    # We create a numpy array of uint8, view as S1, decode to string
    return np.array(consensus_int, dtype=np.uint8).view('S1').tobytes().decode('utf-8')

    

def get_poa_consensus(barcode_series, bias_towards=None):
    """
    Function to call the POA alignment on a Series of Barcode strings.
    
    Args:
        barcode_series (pd.Series): A Series containing all Barcodes for a single cluster.
        
    Returns:
        tuple: (consensus_sequence, multiple_sequence_alignment)
    """
    # Convert the pandas Series of barcodes to a standard Python list
    barcode_list = barcode_series.tolist()

    if bias_towards is not None:
        barcode_list.append(bias_towards)
    
    # Call the external POA function
    _, msa = spoa.poa(barcode_list, algorithm=1)

    consensus = fast_consensus(msa) # use own function as spoa is weird sometimes
    
    return consensus # We will return just the consensus sequence for simplicity in this example

    
def recursive_outlier_removal(all_df, cluster_df, filtered_dist_matrix, expected_error_rate, percentile_th, 
                              verbose,
                             error_rate_multiplier=1.2, n_decoys=50):
    """
    This function assumes that we already have a relatively 'pure' cluster, with maybe a few outliers
    """
    if len(cluster_df) < 2:
        return cluster_df

    m = filtered_dist_matrix.astype(float)
    m[np.triu_indices_from(m)] = np.nan

    max_val, max_idx = np.nanmax(m), np.unravel_index(np.nanargmax(m), m.shape)

    if max_val <= expected_error_rate*error_rate_multiplier:  # increase leniency slightly
        # Then no obvious outliers
        return cluster_df

    # Find all pairings that have a below-average distance from each other
    m = filtered_dist_matrix.astype(float)
    m[np.triu_indices_from(m)] = np.nan
    
    # 2. Calculate the median distance from the valid elements
    median_dist = np.nanmedian(m)

    # Identify largest cluster formed by agg clustering when using median_dist as thresh
    agg = AgglomerativeClustering(
        metric="precomputed",
        # 'single' linkage is often used for identifying outliers/cores
        linkage="single",
        distance_threshold=median_dist,
        # Setting n_clusters=None signals to use the distance_threshold
        n_clusters=None 
    )

    # 4. Fit the model and get cluster labels
    cluster_labels = agg.fit_predict(filtered_dist_matrix)
    
    # 5. Find the largest cluster
    unique, counts = np.unique(cluster_labels, return_counts=True)
    
    # Find the label corresponding to the maximum count
    largest_cluster_label = unique[np.argmax(counts)]
    
    # 6. Filter the original cluster_df 
    
    # Get the row indices of the largest cluster in the original DataFrame
    largest_cluster_indices = np.where(cluster_labels == largest_cluster_label)[0]
    
    # Filter the DataFrame using .iloc (since the indices here are positional)
    largest_cluster_df = cluster_df.iloc[largest_cluster_indices] 

    largest_subclust_dist_matrix = filtered_dist_matrix[np.ix_(largest_cluster_indices, largest_cluster_indices)]

    # Find average distances of each core component to other core components, to help us find the best core
    mean_distances = np.mean(largest_subclust_dist_matrix, axis=1)

    # Find the index of the row with the minimum mean distance
    core_member_index = np.argmin(mean_distances)

    best_core_insert = largest_cluster_df['Insert'].iloc[core_member_index]

    all_ratios = process.cdist([best_core_insert], list(cluster_df['Insert']), scorer=fuzz.ratio, dtype=np.float32)[0]

    outlier_position = np.argmin(all_ratios)  # low ratio = high edit distance
    outlier_ratio = all_ratios[outlier_position]
    outlier_insert = cluster_df['Insert'].iloc[outlier_position]

    # select N random inserts from the all_df
    random_inserts = all_df['Insert'].sample(n=n_decoys)

    # Find distance to all of these
    decoy_ratios = process.cdist([outlier_insert], random_inserts, scorer=fuzz.ratio, dtype=np.float32)[0]

    # if MORE SIMILAR than this, then it's almost certainly a true cluster member
    threshold_ratio = np.percentile(decoy_ratios, percentile_th)

    if outlier_ratio >= threshold_ratio:
        # then it's a member of cluster, don't bother doing again
        return cluster_df  # unchanged
    else:
        # remove this line and run again
        keep_mask = np.arange(len(filtered_dist_matrix)) != outlier_position

        # Filter the distance matrix (remove row AND column)
        new_dist_matrix = filtered_dist_matrix[np.ix_(keep_mask, keep_mask)]
        
        # Filter the DataFrame (remove row)
        new_cluster_df = cluster_df.iloc[keep_mask]
        
        return recursive_outlier_removal(all_df, new_cluster_df, 
                                         new_dist_matrix, 
                                         expected_error_rate, percentile_th, verbose, error_rate_multiplier, n_decoys)
        


def full_analysis(sub_df, all_df, barcode_target, percentile_th, expected_error_rate, 
                             verbose, mi_exclusion_distance, cluster_again_total_mean_thresh=2,
                             cluster_again_single_mean_thresh=3):

    if verbose >= 2:
        print("\n================")
        print(sub_df)

    if len(sub_df) < 2:
        # Ensure consistency by assigning a default cluster if minimal size
        if 'cluster' not in sub_df.columns: sub_df = sub_df.copy(); sub_df['cluster'] = -1
        return sub_df

    
    # Create lists of all barcodes and inserts in sub_df
    barcodes = sub_df['Barcode'].tolist()
    inserts = sub_df['Insert'].tolist()
    
    # =========== Initial processing of barcode-insert pairs ==========================
    # (note, initial clustering is based on error rate, therefore do not subset MSA as
    # need to look at total error)
    msa_int, mi_matrix, msa_subset, dist_matrix_subset, labels, len_msa_inserts = cluster_barcode_insert_pairs(barcodes, 
                                                                                       inserts, 
                                                                                       mi_exclusion_distance,
                                                                                       expected_error_rate,
                                                                                        error_rate_multiplier=2,
                                                                                       verbose=0,
                                                                                        subset_msa=False)

    labels = labels + hash(frozenset(sub_df.index)) % 10_000_000

    if verbose >= 3:
        temp = sub_df.copy()
        temp['cluster'] = labels
        print(temp[['ID', 'cluster']])

    sub_df['cluster'] = labels


    # =========== See if we can break down clusters into reasonable smaller clusters ======

    # 1. Initialize a list to store the split dataframes
    cluster_results = []

    for label in list(set(labels)):
        this_cluster = sub_df[sub_df['cluster'] == label]

        # also filter distance matrix as this will be useful for identifying outliers
        indices = np.where(labels == label)[0]

        filtered_dist_matrix = dist_matrix_subset[np.ix_(indices, indices)]

        final_processed_cluster = recursive_outlier_removal(all_df, this_cluster, filtered_dist_matrix, 
                                                            expected_error_rate, percentile_th, verbose)

        cluster_results.append(final_processed_cluster)

    return pd.concat(cluster_results)


def filter_result_df(result, barcode_target, verbose):
    result = result.copy()    

    # Find consensus barcode of each cluster, and remove those that don't match target
    result['cluster_consensus_bc'] = result.groupby('cluster')['Barcode'].transform(get_poa_consensus)

    if verbose >= 2:
        print(result[['ID', 'cluster_consensus_bc']])
    
    # 2. Apply the final combined filter on - either the consensus matches, or, if only 2 reads, at least 1 matches the target barcode perfectly
    result = result[
        (result['cluster_consensus_bc'] == barcode_target) | # Mask A: Consensus matches target
        ( (result.groupby('cluster')['Barcode'].transform('size') == 2) & # Mask B part 1: Group size is 2
          (result.groupby('cluster')['Barcode'].transform(lambda x: (x == barcode_target).any())) # Mask B part 2: Group contains target
        )
    ]

    # Get consensus insert
    result['cluster_consensus_insert'] = result.groupby('cluster')['Insert'].transform(get_poa_consensus)



    # Re-normalise cluster labels to 0,1,2,... but KEEP -1 as noise
    unique = [u for u in sorted(result['cluster'].unique()) if u != -1]
    mapping = {old: new for new, old in enumerate(unique)}
    result['cluster'] = result['cluster'].map(lambda x: mapping.get(x, -1))

    return result

class MinDistPredictor:
    """
    A class to encapsulate the fitted Inverse Model (Y = a/X + c) 
    for predicting minimum Hamming distance.
    """
    def __init__(self, a: float, c: float):
        """Initializes the predictor with the fitted coefficients."""
        self.a = a
        self.c = c
        self.equation = f"Y = {self.a:.4e}/X + {self.c:.4e}"

    def predict(self, cluster_size):
        """
        Calculates the predicted minimum distance for given cluster size(s).
        
        Args:
            cluster_size (int or list/array): The size(s) of the cluster.
        
        Returns:
            float or array: The predicted minimum distance(s).
        """
        X = np.array(cluster_size)
        return self.a / X + self.c
    
    def get_model_info(self):
        """Prints the model equation."""
        print(f"Model: {self.equation}")



In [4]:
# ==== PARAMS =====
from sklearn.manifold import MDS
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import MDS
import numpy as np
from sklearn.cluster import OPTICS
from sklearn.cluster import SpectralClustering
from scipy.linalg import eigh
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.cluster import AffinityPropagation
import numpy as np
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram
from scipy.optimize import curve_fit

verbose = 1
percentile_th = 95
expected_error_rate = 0.05
PLOTS = False
mi_exclusion_distance=3
n_sims = 150
sim_percentile = 0.1

# ==== RUN =====

barcode_counts = df['Barcode'].value_counts()
all_barcodes = np.array(barcode_counts.index.tolist())     
    

for i, (top_bc, count) in enumerate(barcode_counts.head(1).items()):

    if verbose >= 1:
        print(f"\n\n=============\n{top_bc}")
        print(f"Processing Top Barcode #{i+1}: {top_bc} (Count: {count})")

    # top_bc = "GCAATGCTTGTC" # Force specific barcode if needed for debug
    
    # RapidFuzz
    scores = process.cdist([top_bc], all_barcodes, scorer=fuzz.ratio, dtype=np.float32)[0]
    candidate_bcs = all_barcodes[np.where(scores > 85)[0]]
    filtered_df = df[df['Barcode'].isin(candidate_bcs)].copy()

    # filtered_df = pd.read_csv("~/Downloads/fml.csv")

    if filtered_df.empty: continue

    if verbose >= 2:
        print(f"  > RapidFuzz gathered {len(filtered_df)} reads")

    result_df = full_analysis(filtered_df, df, top_bc, 
                                          percentile_th=percentile_th, verbose=verbose, expected_error_rate=expected_error_rate,
                             mi_exclusion_distance=mi_exclusion_distance)

    result_df2 = filter_result_df(result_df, top_bc, verbose)

    if verbose >= 1:
        print(result_df2[['ID', 'cluster']])



GGTATACATAGT
Processing Top Barcode #1: GGTATACATAGT (Count: 20)
           ID  cluster
125060  20793        0
125061  20793        0
125062  20793        0
125063  20793        0
125064  20793        0
125065  20793        0
125066  20793        0
125067  20793        0
125068  20793        0
121972  20293        1
121973  20293        1
121974  20293        1
121975  20293        1
121976  20293        1
121977  20293        1
121978  20293        1
121979  20293        1
121980  20293        1
121981  20293        1
1745      293        2
1746      293        2
1747      293        2
1748      293        2
1749      293        2
1750      293        2
1751      293        2
1752      293        2
1753      293        2
1754      293        2


In [5]:
result_df2

,ID,Barcode,Insert,cluster,cluster_consensus_bc,cluster_consensus_insert
125060,20793,GGTATACATAGT,AACGCGACGTACCAAAACTACCAACGGGTCTATCAGCAGAATGGAC...,0,GGTATACATAGT,AACGCGACGTACCAAAACTACCAACGGGTCTATCACAGAATGGACT...
125061,20793,GGTATACATAGT,ATCGGACGTACCAAAACTACCAACGGGTCTATCACAGAATGGACTC...,0,GGTATACATAGT,AACGCGACGTACCAAAACTACCAACGGGTCTATCACAGAATGGACT...
125062,20793,GGTATTACATAGT,AACGCGACGTACCAAACATACCAACGGGTCTATCACAGAATGGACT...,0,GGTATACATAGT,AACGCGACGTACCAAAACTACCAACGGGTCTATCACAGAATGGACT...
125063,20793,GGTATACATAGT,AACGCGACGTGCCAAAACTACCAACGGGTCTATCATCAGCATGGAC...,0,GGTATACATAGT,AACGCGACGTACCAAAACTACCAACGGGTCTATCACAGAATGGACT...
125064,20793,GGGTATTACATAGT,GACGAGACGTACCAAAACTACCAACGGGTCTATTACAGAATGGACT...,0,GGTATACATAGT,AACGCGACGTACCAAAACTACCAACGGGTCTATCACAGAATGGACT...
125065,20793,GGTATACATAGT,AACGCGACGTCACCAAATACTCCAACGGGTCTATCACAGAATGGCT...,0,GGTATACATAGT,AACGCGACGTACCAAAACTACCAACGGGTCTATCACAGAATGGACT...
125066,20793,GGTATACATTAGT,AACGCGACGTACCAAAACTTACCAACGGGTCTATCACAGAATGGAC...,0,GGTATACATAGT,AACGCGACGTACCAAAACTACCAACGGGTCTATCACAGAATGGACT...
125067,20793,GGTATACATAGT,AACGCCGACGAACCAAAACTACCAACGGGTCTATCACAGAATGGAC...,0,GGTATACATAGT,AACGCGACGTACCAAAACTACCAACGGGTCTATCACAGAATGGACT...
125068,20793,GGTATACATAGT,AACGCGACGTACCAAAACTACCAACGGGTCTATCAAAGAATGGACT...,0,GGTATACATAGT,AACGCGACGTACCAAAACTACCAACGGGTCTATCACAGAATGGACT...
121972,20293,GGTATACATAGT,CTTGCTCTGACAACAGACATAAACGGAGTAATCTTAGTGCTGTCTG...,1,GGTATACATAGT,CTTGCTCTGACAACAGACATAAACGGAGTAATCTTAGTGCTGTTGG...


In [109]:
result_df2

,ID,Barcode,Insert,cluster,cluster_consensus_bc,cluster_consensus_insert
347,56,TCGCGCACAGGA,TTGGTAACGCTCTGCGACCAATAACGCGGTGCACCAGCGCT,0,TCGCGAACAGGA,TTGGTACGCTCTGCGACCAATAACGCGGTGCATCAGCGCT
348,56,TCGCGACAGA,TTGGTACGCTCTGCGACCAATAACGCGGTGCATCAGCGCA,0,TCGCGAACAGGA,TTGGTACGCTCTGCGACCAATAACGCGGTGCATCAGCGCT
349,56,CCGCGAACAGGA,TTGGTACGCTCTGCGACCAATAACGCGGTGCATCAGCCGCT,0,TCGCGAACAGGA,TTGGTACGCTCTGCGACCAATAACGCGGTGCATCAGCGCT
350,56,TCGCGAACAGGA,TTGGTACGCACTGCGAAATAACGCGGTGTCGTCAGCGTCT,0,TCGCGAACAGGA,TTGGTACGCTCTGCGACCAATAACGCGGTGCATCAGCGCT
351,56,TCGCGAACAGGA,TTGGTACGCTCTGTGACCAATAACGCGGTGCATCACGCT,0,TCGCGAACAGGA,TTGGTACGCTCTGCGACCAATAACGCGGTGCATCAGCGCT
352,56,TCGCTAAACAGGA,TTGGTACGCTTGCGAGCCAATAACGCGATGCATCAGCGCT,0,TCGCGAACAGGA,TTGGTACGCTCTGCGACCAATAACGCGGTGCATCAGCGCT
353,56,TCGCGAACAGGA,TTGGTACGCTGTGCGACAATAACGCGGTGATCAGCGCT,0,TCGCGAACAGGA,TTGGTACGCTCTGCGACCAATAACGCGGTGCATCAGCGCT
354,56,TCGCGAACAGGA,TTGGTACGCTCTCGCGCCAATAACGCGGTGCATCAGCTGCAT,0,TCGCGAACAGGA,TTGGTACGCTCTGCGACCAATAACGCGGTGCATCAGCGCT
355,56,TCGCGAACAGGA,TTGGTCCGCTCTGCGACCAATAACGCAGTGCATCAGCCT,0,TCGCGAACAGGA,TTGGTACGCTCTGCGACCAATAACGCGGTGCATCAGCGCT
356,56,TCGCGAACAGGA,TTGGTACTCTCTACGACCAATAACGCGGTGAATCTGTGCT,0,TCGCGAACAGGA,TTGGTACGCTCTGCGACCAATAACGCGGTGCATCAGCGCT
